# SolarSpine R2B — reproduction notebook for jury

**Method:** YOLO26n-seg A3 + native-ROI residual U-Net R2A. No training, no TEST ground-truth reading, and no Kaggle upload. All inference code and both trained weights must come from the publicly accessible release repository. Requires official competition TEST JPEGs provided by the reviewer and a Tesla T4 environment matching recorded versions.

The source code and masks are checked by SHA256; outputs use native 2048 × 2048 compressed COCO RLE. The entire prediction pipeline is invoked from this notebook. It does **not** depend on the owner's private Google Drive or on a separate source ZIP.


In [ ]:
# Edit REPO_URL only after the owner publishes the approved jury repository.
from pathlib import Path
import os, sys, subprocess, json, hashlib
REPO_URL = ''  # e.g. https://github.com/OWNER/solarspine-r2b-final-2026.git
REPO_DIR = Path('/content/solarspine-r2b-final-2026')
IMAGES = Path('/content/official_test/test_images')   # official TEST JPEG folder
OUTPUT_DIR = Path('/content/solarspine_r2b_new_run')    # must be new/external to inputs
assert REPO_URL.startswith('https://github.com/') and REPO_URL.endswith('.git'), 'Set public REPO_URL first'
assert IMAGES.is_dir(), 'Upload/mount the official competition TEST JPEGs'
print('CONFIG_READY')


In [ ]:
# Clone the PUBLIC code and checkpoint repository. Never use a GitHub token or private checkout.
if not (REPO_DIR / 'README.md').is_file():
    assert not REPO_DIR.exists(), 'REPO_DIR exists but is not a complete checkout'
    subprocess.run(['git','clone','--depth','1',REPO_URL,str(REPO_DIR)],check=True)
assert (REPO_DIR/'.gitattributes').is_file(), 'The release repository does not have expected LFS configuration'
subprocess.run(['git','-C',str(REPO_DIR),'lfs','pull'],check=True)
subprocess.run([sys.executable,str(REPO_DIR/'scripts/preflight_release.py'),'--strict'],check=True)
print('SOURCE_AND_WEIGHTS_VERIFIED')


In [ ]:
# Verify the exact accepted CUDA/T4 torch runtime BEFORE any dependent packages are installed.
assert sys.version_info[:2] == (3,13), 'Recorded inference requires Python 3.13'
import torch, torchvision
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Select NVIDIA Tesla T4'
assert torch.__version__=='2.11.0+cu130' and torch.version.cuda=='13.0', 'Torch/CUDA version differs from accepted runtime'
assert torchvision.__version__=='0.26.0+cu130', 'Torchvision version differs from accepted runtime'
VENV=Path('/content/solarspine_r2b_venv')
PY=VENV/'bin/python'
if not PY.exists(): subprocess.run([sys.executable,'-m','venv','--system-site-packages','--without-pip',str(VENV)],check=True)
requirements=['numpy==2.1.3','Pillow==11.3.0','pycocotools==2.0.11',
              'scipy==1.16.3','opencv-python==4.12.0.88','ultralytics==8.4.177']
subprocess.run([sys.executable,'-m','pip','--python',str(PY),'install','--disable-pip-version-check','--no-deps',*requirements],check=True)
print('ACCEPTED_RUNTIME_PREPARED')


In [ ]:
# Two independent prediction processes, using only official JPEG and frozen weights.
DETECTOR=REPO_DIR/'models/detector/best.pt'
REFINER=REPO_DIR/'models/refiner'
command=[str(PY),str(REPO_DIR/'delivery/r2b/run_inference.py'),
         '--images',str(IMAGES),'--detector-weights',str(DETECTOR),
         '--refiner-dir',str(REFINER),'--output-dir',str(OUTPUT_DIR)]
subprocess.run(command,check=True)
report=json.loads((OUTPUT_DIR/'summary.json').read_text())
assert report['complete'] is True and report['GT_used'] is False and report['smoke'] is False
assert report['stages']['detector']['images']==report['stages']['refiner']['images']==180
print('JURY_INFERENCE_COMPLETE_NO_KAGGLE_UPLOAD',str(OUTPUT_DIR/'submission_r2b.csv'))


In [ ]:
# Output consistency check: not a substitute for a private leaderboard score.
def sha(p):
 d=hashlib.sha256()
 with Path(p).open('rb') as f:
  for b in iter(lambda:f.read(1024*1024),b''):d.update(b)
 return d.hexdigest()
expected={'detector.csv':'347307dc95727dbd9ea97d9d474c2125eb5bb14515e79c8bce563e51ead61a39',
          'submission_r2b.csv':'c241bda04c1fefdab5fe1b6b6d90f4aeac23a02a2003f830c0b9b1b289d3294f'}
for name,h in expected.items():
 got=sha(OUTPUT_DIR/name)
 print(name,got,'MATCH' if got==h else 'DIFFERENT')
 assert got==h, f'Prediction bytes differ from accepted R2B: {name}'
print('SOLARSPINE_FINAL_CSV_REPRODUCED_EXACTLY')
